In [2]:
import kagglehub
import os
import pandas as pd

path = kagglehub.dataset_download("naserabdullahalam/phishing-email-dataset")

print(path)
print(os.listdir(path))

100%|█████████████████████████████████████| 77.1M/77.1M [00:06<00:00, 11.6MB/s]

Extracting files...


/Users/aarushisingh/.cache/kagglehub/datasets/naserabdullahalam/phishing-email-dataset/versions/1
['Nigerian_Fraud.csv', 'Ling.csv', 'Nazario.csv', 'SpamAssasin.csv', 'CEAS_08.csv', 'phishing_email.csv', 'Enron.csv']


In [8]:
df = pd.read_csv(os.path.join(path, "phishing_email.csv"))
df.head()
df.info()
df["label"].value_counts()

<class 'pandas.DataFrame'>
RangeIndex: 82486 entries, 0 to 82485
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   text_combined  82486 non-null  str  
 1   label          82486 non-null  int64
dtypes: int64(1), str(1)
memory usage: 1.3 MB


label
1    42891
0    39595
Name: count, dtype: int64

In [9]:
from sklearn.model_selection import train_test_split

X = df["text_combined"].fillna("")
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    ngram_range=(1,2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [13]:
#Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

lr = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr.fit(X_train_tfidf, y_train)

y_pred = lr.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9838162201478967
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      7919
           1       0.98      0.99      0.98      8579

    accuracy                           0.98     16498
   macro avg       0.98      0.98      0.98     16498
weighted avg       0.98      0.98      0.98     16498

[[7772  147]
 [ 120 8459]]


In [14]:
#Random Forest
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_tfidf, y_train)

y_pred_rf = rf.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))

Accuracy: 0.9871499575706146
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      7919
           1       0.99      0.99      0.99      8579

    accuracy                           0.99     16498
   macro avg       0.99      0.99      0.99     16498
weighted avg       0.99      0.99      0.99     16498

[[7810  109]
 [ 103 8476]]


In [15]:
#Multinomial Naive Bayes
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()

nb.fit(X_train_tfidf, y_train)

y_pred_nb = nb.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))
print(confusion_matrix(y_test, y_pred_nb))

Accuracy: 0.9599951509273852
              precision    recall  f1-score   support

           0       0.93      0.99      0.96      7919
           1       0.99      0.94      0.96      8579

    accuracy                           0.96     16498
   macro avg       0.96      0.96      0.96     16498
weighted avg       0.96      0.96      0.96     16498

[[7806  113]
 [ 547 8032]]


In [ ]:
#MLP
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(
    hidden_layer_sizes=(100,),
    max_iter=300,
    random_state=42
)

mlp.fit(X_train_tfidf, y_train)

y_pred_mlp = mlp.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred_mlp))
print(classification_report(y_test, y_pred_mlp))
print(confusion_matrix(y_test, y_pred_mlp))

In [ ]:
results = {
    "Logistic Regression": accuracy_score(y_test, y_pred),
    "Random Forest": accuracy_score(y_test, y_pred_rf),
    "Naive Bayes": accuracy_score(y_test, y_pred_nb),
    "MLP": accuracy_score(y_test, y_pred_mlp)
}

for model, acc in results.items():
    print(f"{model}: {acc:.4f}")